# 02 — Data Quality, Cleaning & Feature Engineering

This notebook validates the raw weather data loaded into DuckDB, applies cleaning logic, builds model-ready features, and stores the final feature table in DuckDB.

The output of this notebook is:

`analytics.model_features`

This table is used by the modeling notebook and by the automated pipeline.

## 1. Imports & Setup


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.db import run_query, get_connection

from src.quality_checks import (
    check_missing_values,
    check_duplicate_rows,
    check_duplicate_city_dates,
    check_date_coverage,
    check_missing_dates,
    check_column_consistency,
    check_weather_ranges,
)

from src.cleaning import clean_data

from src.features import (
    build_features,
    get_feature_columns,
    get_target_columns,
)

## 2. Load Raw Data from DuckDB


In [2]:
historical_df = run_query("SELECT * FROM raw.historical").copy()
forecast_df = run_query("SELECT * FROM raw.forecast").copy()

historical_df["time"] = pd.to_datetime(historical_df["time"])
forecast_df["time"] = pd.to_datetime(forecast_df["time"])

print("Historical shape:", historical_df.shape)
print("Forecast shape:", forecast_df.shape)

display(historical_df.head())
display(forecast_df.head())

Historical shape: (10960, 9)
Forecast shape: (35, 9)


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city
0,2020-04-27,19.3,0.0,15.9,78,26,19.8,45068.90,Baku
1,2020-04-28,14.6,0.0,42.0,83,71,11.9,38436.84,Baku
2,2020-04-29,17.6,0.0,22.4,61,50,16.3,39311.92,Baku
3,2020-04-30,18.0,0.0,23.0,77,50,16.0,44363.33,Baku
4,2020-05-01,19.5,0.0,20.2,78,24,18.5,46341.87,Baku


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city
0,2026-04-28,14.1,0.0,45.2,52,90,7.1,31968.01,Baku
1,2026-04-29,14.3,0.0,22.1,63,96,10.5,15011.54,Baku
2,2026-04-30,17.7,0.0,19.1,59,42,15.3,42871.36,Baku
3,2026-05-01,18.0,0.0,31.0,76,20,14.7,48670.33,Baku
4,2026-05-02,19.2,0.0,32.3,80,63,16.6,46724.45,Baku


## 3. Historical Data Quality Checks


In [3]:
print("Historical data quality checks")

print("\nMissing values:")
display(check_missing_values(historical_df))

print("\nDuplicate rows:")
print(check_duplicate_rows(historical_df))

print("\nDuplicate city-date records:")
print(check_duplicate_city_dates(historical_df))

print("\nWeather range violations:")
print(check_weather_ranges(historical_df))

Historical data quality checks

Missing values:


{'check': 'missing_values',
 'status': 'PASS',
 'details': {'time': 0,
  'temperature_2m_max': 0,
  'precipitation_sum': 0,
  'wind_speed_10m_max': 0,
  'relative_humidity_2m_mean': 0,
  'cloud_cover_mean': 0,
  'apparent_temperature_max': 0,
  'sunshine_duration': 0,
  'city': 0}}


Duplicate rows:
{'check': 'duplicate_rows', 'status': 'PASS', 'details': '0 duplicates'}

Duplicate city-date records:
{'check': 'duplicate_city_dates', 'status': 'PASS', 'details': '0 duplicates'}

Weather range violations:
{'check': 'weather_ranges', 'status': 'PASS', 'details': {'temperature_2m_max': 0, 'precipitation_sum': 0, 'relative_humidity_2m_mean': 0, 'apparent_temperature_max': 0}}


## 4. Forecast Data Quality Checks


In [4]:
print("Forecast data quality checks")

print("\nMissing values:")
display(check_missing_values(forecast_df))

print("\nDuplicate rows:")
print(check_duplicate_rows(forecast_df))

print("\nDuplicate city-date records:")
print(check_duplicate_city_dates(forecast_df))

print("\nWeather range violations:")
print(check_weather_ranges(forecast_df))

Forecast data quality checks

Missing values:


{'check': 'missing_values',
 'status': 'PASS',
 'details': {'time': 0,
  'temperature_2m_max': 0,
  'precipitation_sum': 0,
  'wind_speed_10m_max': 0,
  'relative_humidity_2m_mean': 0,
  'cloud_cover_mean': 0,
  'apparent_temperature_max': 0,
  'sunshine_duration': 0,
  'city': 0}}


Duplicate rows:
{'check': 'duplicate_rows', 'status': 'PASS', 'details': '0 duplicates'}

Duplicate city-date records:
{'check': 'duplicate_city_dates', 'status': 'PASS', 'details': '0 duplicates'}

Weather range violations:
{'check': 'weather_ranges', 'status': 'PASS', 'details': {'temperature_2m_max': 0, 'precipitation_sum': 0, 'relative_humidity_2m_mean': 0, 'apparent_temperature_max': 0}}


## 5. Coverage and Consistency Checks


In [5]:
print("Historical city/date coverage:")
display(check_date_coverage(historical_df))

print("\nHistorical missing dates by city:")
print(check_missing_dates(historical_df))

print("\nHistorical city counts:")
display(historical_df["city"].value_counts())

print("\nForecast city counts:")
display(forecast_df["city"].value_counts())

print("\nHistorical vs forecast column consistency:")
check_column_consistency(historical_df, forecast_df)

Historical city/date coverage:


{'check': 'date_coverage',
 'status': 'PASS',
 'details': {'min': {'Baku': Timestamp('2020-04-27 00:00:00'),
   'Gabala': Timestamp('2020-04-27 00:00:00'),
   'Guba': Timestamp('2020-04-27 00:00:00'),
   'Lankaran': Timestamp('2020-04-27 00:00:00'),
   'Shaki': Timestamp('2020-04-27 00:00:00')},
  'max': {'Baku': Timestamp('2026-04-27 00:00:00'),
   'Gabala': Timestamp('2026-04-27 00:00:00'),
   'Guba': Timestamp('2026-04-27 00:00:00'),
   'Lankaran': Timestamp('2026-04-27 00:00:00'),
   'Shaki': Timestamp('2026-04-27 00:00:00')},
  'count': {'Baku': 2192,
   'Gabala': 2192,
   'Guba': 2192,
   'Lankaran': 2192,
   'Shaki': 2192}}}


Historical missing dates by city:
{'check': 'missing_dates', 'status': 'PASS', 'details': {'Baku': 0, 'Gabala': 0, 'Guba': 0, 'Lankaran': 0, 'Shaki': 0}}

Historical city counts:


city
Baku        2192
Gabala      2192
Guba        2192
Lankaran    2192
Shaki       2192
Name: count, dtype: int64


Forecast city counts:


city
Baku        7
Gabala      7
Guba        7
Lankaran    7
Shaki       7
Name: count, dtype: int64


Historical vs forecast column consistency:


{'check': 'column_consistency',
 'status': 'PASS',
 'details': {'same_columns': True, 'only_in_first': [], 'only_in_second': []}}

## 6. Data Quality Summary

The historical and forecast datasets were checked for missing values, duplicate rows, duplicate city-date records, date coverage, city coverage, column consistency, and unrealistic weather values.

The checks confirm that the raw data is complete, consistent, and suitable for cleaning and feature engineering.

## 7. Clean Historical Data


### 7.1 — apply cleaning

In [6]:
clean_df = clean_data(historical_df)

print("Raw historical shape:", historical_df.shape)
print("Clean historical shape:", clean_df.shape)

clean_df.head()

Raw historical shape: (10960, 9)
Clean historical shape: (10960, 9)


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city
0,2020-04-27,19.3,0.0,15.9,78,26,19.8,45068.90,Baku
1,2020-04-28,14.6,0.0,42.0,83,71,11.9,38436.84,Baku
2,2020-04-29,17.6,0.0,22.4,61,50,16.3,39311.92,Baku
3,2020-04-30,18.0,0.0,23.0,77,50,16.0,44363.33,Baku
4,2020-05-01,19.5,0.0,20.2,78,24,18.5,46341.87,Baku


### 7.2 — validate cleaned data

In [7]:
print("Cleaned data validation")

print("\nMissing values:")
display(clean_df.isna().sum())

print("\nDuplicate rows:")
print(clean_df.duplicated().sum())

print("\nDuplicate city-date records:")
print(clean_df.duplicated(subset=["city", "time"]).sum())

print("\nDate continuity check:")
display(clean_df.groupby("city")["time"].diff().value_counts())

Cleaned data validation

Missing values:


time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64


Duplicate rows:
0

Duplicate city-date records:
0

Date continuity check:


time
1 days    10955
Name: count, dtype: int64

## 8. Feature Engineering


### 8.1 — build features

In [8]:
feature_df, city_encoder = build_features(clean_df)

print("Feature dataframe shape:", feature_df.shape)
feature_df.head()

Feature dataframe shape: (10955, 20)


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city,month,day_of_month,temperature_lag_1,precipitation_lag_1,wind_lag_1,humidity_lag_1,temperature_3d_avg,precipitation_7d_sum,wind_3d_avg,humidity_7d_avg,city_encoded
0,2020-04-28,14.6,0.0,42.0,83,71,11.9,38436.84,Baku,4,28,19.3,0.0,15.9,78.0,16.950000,0.0,28.950000,80.500000,0
1,2020-04-29,17.6,0.0,22.4,61,50,16.3,39311.92,Baku,4,29,14.6,0.0,42.0,83.0,17.166667,0.0,26.766667,74.000000,0
2,2020-04-30,18.0,0.0,23.0,77,50,16.0,44363.33,Baku,4,30,17.6,0.0,22.4,61.0,16.733333,0.0,29.133333,74.750000,0
3,2020-05-01,19.5,0.0,20.2,78,24,18.5,46341.87,Baku,5,1,18.0,0.0,23.0,77.0,18.366667,0.0,21.866667,75.400000,0
4,2020-05-02,19.6,0.0,20.7,83,45,19.0,43993.52,Baku,5,2,19.5,0.0,20.2,78.0,19.033333,0.0,21.300000,76.666667,0


### 8.2 — inspect features and targets

In [9]:
feature_cols = get_feature_columns()
target_cols = get_target_columns()

print("Feature columns:")
for col in feature_cols:
    print(f"- {col}")

print("\nTarget columns:")
for col in target_cols:
    print(f"- {col}")

print("\nMissing values after feature engineering:")
display(feature_df[feature_cols + target_cols].isna().sum())

Feature columns:
- apparent_temperature_max
- sunshine_duration
- city_encoded
- month
- day_of_month
- temperature_lag_1
- precipitation_lag_1
- wind_lag_1
- humidity_lag_1
- temperature_3d_avg
- precipitation_7d_sum
- wind_3d_avg
- humidity_7d_avg

Target columns:
- temperature_2m_max
- precipitation_sum
- wind_speed_10m_max
- relative_humidity_2m_mean
- cloud_cover_mean

Missing values after feature engineering:


apparent_temperature_max     0
sunshine_duration            0
city_encoded                 0
month                        0
day_of_month                 0
temperature_lag_1            0
precipitation_lag_1          0
wind_lag_1                   0
humidity_lag_1               0
temperature_3d_avg           0
precipitation_7d_sum         0
wind_3d_avg                  0
humidity_7d_avg              0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
dtype: int64

## 9. Store Model-Ready Dataset in DuckDB


### 9.1 — store table

In [10]:
with get_connection() as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS analytics;")
    conn.register("feature_df_view", feature_df)

    conn.execute("""
        CREATE OR REPLACE TABLE analytics.model_features AS
        SELECT * FROM feature_df_view;
    """)

print("analytics.model_features table created.")

analytics.model_features table created.


### 9.2 — verify table

In [11]:
model_features_summary = run_query("""
SELECT
    MIN(time) AS min_date,
    MAX(time) AS max_date,
    COUNT(*) AS rows
FROM analytics.model_features
""")

display(model_features_summary)

run_query("""
SELECT *
FROM analytics.model_features
LIMIT 5
""")

,min_date,max_date,rows
0,2020-04-28,2026-04-27,10955


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city,month,day_of_month,temperature_lag_1,precipitation_lag_1,wind_lag_1,humidity_lag_1,temperature_3d_avg,precipitation_7d_sum,wind_3d_avg,humidity_7d_avg,city_encoded
0,2020-04-28,14.6,0.0,42.0,83,71,11.9,38436.84,Baku,4,28,19.3,0.0,15.9,78.0,16.950000,0.0,28.950000,80.500000,0
1,2020-04-29,17.6,0.0,22.4,61,50,16.3,39311.92,Baku,4,29,14.6,0.0,42.0,83.0,17.166667,0.0,26.766667,74.000000,0
2,2020-04-30,18.0,0.0,23.0,77,50,16.0,44363.33,Baku,4,30,17.6,0.0,22.4,61.0,16.733333,0.0,29.133333,74.750000,0
3,2020-05-01,19.5,0.0,20.2,78,24,18.5,46341.87,Baku,5,1,18.0,0.0,23.0,77.0,18.366667,0.0,21.866667,75.400000,0
4,2020-05-02,19.6,0.0,20.7,83,45,19.0,43993.52,Baku,5,2,19.5,0.0,20.2,78.0,19.033333,0.0,21.300000,76.666667,0


## 10. Final Summary

The raw historical data was validated, cleaned, and transformed into a model-ready feature table.

The final dataset includes:

- original target weather variables
- calendar features
- lag features
- rolling trend features
- encoded city information

The prepared dataset was stored in DuckDB as:

`analytics.model_features`

This table is now ready for model training and evaluation.